In [1]:
import numpy as np 
from scipy.interpolate import CubicSpline

def prolong(strokes: list[list[tuple[float, float]]]) -> list[list[tuple[float, float]]]:
    """
    Extend each stroke by adding one extrapolated point before the first point
    and one after the last point, with the extension distance based on the
    average distance between consecutive points in the stroke.
    
    Args:
        prototype: List of strokes, where each stroke is a list of (x, y) tuples.
    
    Returns:
        prolonged_prototype: Modified prototype with extended strokes.
    """
    prolonged_strokes = []
    
    for stroke in strokes:
        if len(stroke) < 3:
            prolonged_strokes.append(stroke)
            continue  # Not enough points to fit a spline

        points = np.array(stroke)
        x = points[:, 0]
        y = points[:, 1]

        # Calculate differences between consecutive points
        dx = np.diff(x)
        dy = np.diff(y)
        
        # Compute distances between consecutive points
        distances = np.hypot(dx, dy)
        
        # Calculate the average distance
        avg_distance = np.mean(distances) * 0.3
        
        # Parameter t based on cumulative distance
        t = np.insert(np.cumsum(distances), 0, 0)

        # Remove duplicate t values to satisfy the CubicSpline requirement
        t, unique_idx = np.unique(t, return_index=True)
        x = x[unique_idx]
        y = y[unique_idx]

        if len(x) < 3:
            prolonged_strokes.append(stroke)
            continue  # Not enough unique points to fit a spline
        
        # Fit cubic splines
        cs_x = CubicSpline(t, x, bc_type='natural')
        cs_y = CubicSpline(t, y, bc_type='natural')
        
        # Extrapolate one point before the first point
        t_before = t[0] - avg_distance
        x_before = cs_x(t_before)
        y_before = cs_y(t_before)
        
        # Extrapolate one point after the last point
        t_after = t[-1] + avg_distance
        x_after = cs_x(t_after)
        y_after = cs_y(t_after)
        
        # Construct the extended stroke
        extended_stroke = [(x_before, y_before)] + stroke + [(x_after, y_after)]
        prolonged_strokes.append(extended_stroke)
    
    return prolonged_strokes
    

In [14]:
fkt = lambda x: x ** 2 + 3
spline = [(x, fkt(x)) for x in np.linspace(-3, 3, 5)]
print(spline)
print(prolong([spline, spline, [(x, x+2) for x in np.linspace(10, 30, 5)]]))

[(np.float64(-3.0), np.float64(12.0)), (np.float64(-1.5), np.float64(5.25)), (np.float64(0.0), np.float64(3.0)), (np.float64(1.5), np.float64(5.25)), (np.float64(3.0), np.float64(12.0))]
[[(array(-3.14545185), array(13.26340834)), (np.float64(-3.0), np.float64(12.0)), (np.float64(-1.5), np.float64(5.25)), (np.float64(0.0), np.float64(3.0)), (np.float64(1.5), np.float64(5.25)), (np.float64(3.0), np.float64(12.0)), (array(3.14545185), array(13.26340834))], [(array(-3.14545185), array(13.26340834)), (np.float64(-3.0), np.float64(12.0)), (np.float64(-1.5), np.float64(5.25)), (np.float64(0.0), np.float64(3.0)), (np.float64(1.5), np.float64(5.25)), (np.float64(3.0), np.float64(12.0)), (array(3.14545185), array(13.26340834))], [(array(8.5), array(10.5)), (np.float64(10.0), np.float64(12.0)), (np.float64(15.0), np.float64(17.0)), (np.float64(20.0), np.float64(22.0)), (np.float64(25.0), np.float64(27.0)), (np.float64(30.0), np.float64(32.0)), (array(31.5), array(33.5))]]
